# pysentimiento — clasificación completa DNU (259 752 tweets)

Aplica el analizador de hate speech de **pysentimiento** (`RoBERTuito`) al dataset completo.

**Antes de ejecutar:**
1. Seleccionar entorno **GPU** en *Entorno de ejecución → Cambiar tipo de entorno de ejecución*
2. Subir el archivo `tweets_dnu_clasificados.csv` a Colab (panel *Archivos*, o desde Drive)
3. Ajustar `INPUT_CSV` en la celda de configuración si el nombre difiere

El notebook guarda un checkpoint cada 10 000 filas; si la sesión se corta se puede retomar.

## 1. Verificar GPU

In [ ]:
import torch

device = 0 if torch.cuda.is_available() else -1
device_name = torch.cuda.get_device_name(0) if device == 0 else "CPU (sin GPU)"
print(f"Dispositivo: {device_name}")
if device == -1:
    print("⚠️  Sin GPU — el procesamiento será muy lento. Cambiar el entorno de ejecución.")

## 2. Instalación

In [ ]:
!pip install -q pysentimiento torch pandas tqdm

## 3. Imports

In [ ]:
import os
import pandas as pd
from tqdm.auto import tqdm
from pysentimiento import create_analyzer

## 4. Configuración

In [ ]:
INPUT_CSV        = 'tweets_dnu_clasificados.csv'
OUTPUT_CSV       = 'DNU_pysentiment_full.csv'
CHECKPOINT_CSV   = 'DNU_pysentiment_full_checkpoint.csv'

TEXT_COLUMN      = 'text'
MAX_ROWS         = None   # None = todos los tweets
BATCH_SIZE       = 64     # bajar a 32 si hay OOM
CHECKPOINT_EVERY = 10_000 # guardar progreso cada N filas

## 5. Cargar datos

In [ ]:
df = pd.read_csv(INPUT_CSV, low_memory=False)
if MAX_ROWS:
    df = df.head(MAX_ROWS)
df = df.reset_index(drop=True)

print(f'Filas a procesar: {len(df):,}')
print(f'Columnas: {list(df.columns)}')
df[[TEXT_COLUMN]].head(3)

## 6. Cargar analizador pysentimiento

In [ ]:
analyzer = create_analyzer(task='hate_speech', lang='es')
print('Analizador cargado correctamente.')

## 7. Prueba rápida con la primera fila

In [ ]:
sample_text = str(df[TEXT_COLUMN].iloc[0])
print('Texto:', sample_text)

pred = analyzer.predict(sample_text)
print('output:  ', pred.output)
print('probas:  ', pred.probas)

## 8. Clasificar todos los tweets

- `raw_label`: lista de categorías detectadas (`[]` = sin odio)
- `binary_pred`: `1` = odio, `0` = sin odio
- `raw_probas`: probabilidades crudas de cada categoría
- `score`: probabilidad máxima entre las categorías positivas (o `None` si no hay)

El progreso se guarda en `CHECKPOINT_CSV` cada `CHECKPOINT_EVERY` filas.

In [ ]:
texts  = df[TEXT_COLUMN].fillna('').tolist()
n      = len(texts)

raw_labels   = [None] * n
scores       = [None] * n
binary_preds = [None] * n
raw_probas   = [None] * n

# Retomar desde checkpoint si existe
start_i = 0
if os.path.exists(CHECKPOINT_CSV):
    ck = pd.read_csv(CHECKPOINT_CSV)
    done = ck['pysentimiento_hate__binary_pred'].notna().sum()
    if done > 0:
        start_i = int(done)
        raw_labels[:start_i]   = ck['pysentimiento_hate__raw_label'].tolist()[:start_i]
        scores[:start_i]       = ck['pysentimiento_hate__score'].tolist()[:start_i]
        binary_preds[:start_i] = ck['pysentimiento_hate__binary_pred'].tolist()[:start_i]
        raw_probas[:start_i]   = ck['pysentimiento_hate__raw_probas'].tolist()[:start_i]
        print(f'Retomando desde fila {start_i:,} (checkpoint encontrado)')

for i in tqdm(range(start_i, n, BATCH_SIZE), desc='pysentimiento_hate'):
    batch = texts[i:i + BATCH_SIZE]
    preds = analyzer.predict(batch)

    for j, pred in enumerate(preds):
        idx          = i + j
        output_label = getattr(pred, 'output', None)
        probas       = getattr(pred, 'probas', None)

        if isinstance(output_label, list):
            binary = 1 if len(output_label) > 0 else 0
            # score: max prob entre las categorías positivas detectadas
            if probas and len(output_label) > 0:
                score_val = max(probas.get(k, 0) for k in output_label)
            else:
                score_val = None
        else:
            s = str(output_label).strip().lower() if output_label is not None else ''
            binary    = 1 if s in {'hateful', 'hate'} else (0 if s in {'non-hateful', 'non_hateful'} else None)
            score_val = None

        raw_labels[idx]   = str(output_label)
        scores[idx]       = score_val
        binary_preds[idx] = binary
        raw_probas[idx]   = str(probas) if probas is not None else None

    # Checkpoint
    if (i + BATCH_SIZE) % CHECKPOINT_EVERY < BATCH_SIZE:
        ck_df = df.copy()
        ck_df['pysentimiento_hate__raw_label']   = raw_labels
        ck_df['pysentimiento_hate__score']       = scores
        ck_df['pysentimiento_hate__binary_pred'] = binary_preds
        ck_df['pysentimiento_hate__raw_probas']  = raw_probas
        ck_df.to_csv(CHECKPOINT_CSV, index=False)
        print(f'  checkpoint: {i + BATCH_SIZE:,} / {n:,} filas')

# Guardar resultado final
df['pysentimiento_hate__raw_label']   = raw_labels
df['pysentimiento_hate__score']       = scores
df['pysentimiento_hate__binary_pred'] = binary_preds
df['pysentimiento_hate__raw_probas']  = raw_probas

df.to_csv(OUTPUT_CSV, index=False)
print(f'\nGuardado: {OUTPUT_CSV}  —  {len(df):,} filas')

## 9. Resumen de resultados

In [ ]:
pred_col = 'pysentimiento_hate__binary_pred'
valid    = pd.to_numeric(df[pred_col], errors='coerce')

n_total  = len(df)
n_valid  = int(valid.notna().sum())
n_hate   = int((valid == 1).sum())
n_no_hate = int((valid == 0).sum())

print(f'Total filas:      {n_total:>8,}')
print(f'Predicciones OK:  {n_valid:>8,}  ({n_valid/n_total*100:.1f}%)')
print(f'  Odio (1):       {n_hate:>8,}  ({n_hate/n_valid*100:.1f}%)')
print(f'  Sin odio (0):   {n_no_hate:>8,}  ({n_no_hate/n_valid*100:.1f}%)')

## 10. Vista previa

In [ ]:
cols_preview = ['id', TEXT_COLUMN,
                'pysentimiento_hate__raw_label',
                'pysentimiento_hate__score',
                'pysentimiento_hate__binary_pred',
                'pysentimiento_hate__raw_probas']
cols_preview = [c for c in cols_preview if c in df.columns]
df[cols_preview].head(10)

## 11. Ejemplos de tweets marcados como odio

In [ ]:
hate_df = df[df['pysentimiento_hate__binary_pred'] == 1]
print(f'Tweets con odio: {len(hate_df):,}')
hate_df[['id', TEXT_COLUMN, 'pysentimiento_hate__raw_label', 'pysentimiento_hate__raw_probas']].head(15)

## 12. Descargar resultado

In [ ]:
from google.colab import files
files.download(OUTPUT_CSV)